In [5]:
import math
from itertools import product

In [6]:
r = 0.0132
T = 0.25
dt = 1 / 252

S1_0 = 158.12
S2_0 = 170.33

sigma1_0 = 0.2214
sigma2_0 = 0.3036
rho_0 = 0.237

In [7]:
def option_value(t, s1, s2, sigma1, sigma2, rho):
    tau = T - t

    return math.exp(r * tau) * (
        s1**2 * math.exp(sigma1**2 * tau)
        - 2 * s1 * s2
          * math.exp(sigma1 * sigma2 * rho * tau)
        + s2**2 * math.exp(sigma2**2 * tau)
    )

def option_delta(t, s1, s2, sigma1, sigma2, rho):
    tau = T - t

    h1 = math.exp(r * tau) * (
        2 * s1 * math.exp(sigma1**2 * tau)
        - 2 * s2
          * math.exp(sigma1 * sigma2 * rho * tau)
    )

    h2 = math.exp(r * tau) * (
        -2 * s1
         * math.exp(sigma1 * sigma2 * rho * tau)
        + 2 * s2 * math.exp(sigma2**2 * tau)
    )

    return h1, h2

In [8]:
V0 = option_value(
    0, S1_0, S2_0,
    sigma1_0, sigma2_0, rho_0
)

h1, h2 = option_delta(
    0, S1_0, S2_0,
    sigma1_0, sigma2_0, rho_0
)

portfolio_0 = V0 - h1 * S1_0 - h2 * S2_0

log_returns = [-0.20, -0.10, -0.05,
                0.05, 0.10, 0.20]

vol_multipliers = [0.5, 0.75, 1.25,
                   1.5, 1.75, 2.0]

rho_changes = [-0.5, 0.0, 0.5]

t5 = 5 * dt

max_loss = -math.inf
worst_scenario = None

In [9]:
for x1, x2, m1, m2, drho in product(
    log_returns,
    log_returns,
    vol_multipliers,
    vol_multipliers,
    rho_changes
):
    S1_5 = S1_0 * math.exp(x1)
    S2_5 = S2_0 * math.exp(x2)

    sigma1_5 = sigma1_0 * m1
    sigma2_5 = sigma2_0 * m2

    rho_5 = rho_0 + drho

    V5 = option_value(
        t5,
        S1_5,
        S2_5,
        sigma1_5,
        sigma2_5,
        rho_5
    )

    portfolio_5 = (
        V5
        - h1 * S1_5
        - h2 * S2_5
    )

    loss = -(portfolio_5 - portfolio_0)

    if loss > max_loss:
        max_loss = loss

        worst_scenario = (
            x1, x2,
            m1, m2,
            drho
        )

print("Maximum loss:", max_loss)
print("Worst scenario:", worst_scenario)

Maximum loss: 1000.3617172118802
Worst scenario: (0.2, 0.2, 0.5, 0.5, 0.5)


In [11]:
return_weights = {
    -0.20: 0.5,
    -0.10: 0.75,
    -0.05: 1.0,
     0.05: 1.0,
     0.10: 0.75,
     0.20: 0.5
}

# Weights for volatility multipliers
vol_weights = {
    0.5: 0.5,
    0.75: 1.25,
    1.25: 0.75,
    1.5: 0.75,
    1.75: 1.25,
    2.0: 0.5
}

# Weights for correlation changes
rho_weights = {
    -0.5: 1.25,
     0.0: 0.5,
     0.5: 1.25
}

max_weighted_loss = -math.inf
worst_weighted_scenario = None
worst_portfolio_loss = None
worst_weight = None

In [13]:
for x1, x2, m1, m2, drho in product(
    log_returns,
    log_returns,
    vol_multipliers,
    vol_multipliers,
    rho_changes
):
    S1_5 = S1_0 * math.exp(x1)
    S2_5 = S2_0 * math.exp(x2)

    sigma1_5 = sigma1_0 * m1
    sigma2_5 = sigma2_0 * m2

    rho_5 = rho_0 + drho

    V5 = option_value(
        t5,
        S1_5,
        S2_5,
        sigma1_5,
        sigma2_5,
        rho_5
    )

    portfolio_5 = (
        V5
        - h1 * S1_5
        - h2 * S2_5
    )

    loss = -(portfolio_5 - portfolio_0)

    weight = (
        return_weights[x1]
        * return_weights[x2]
        * vol_weights[m1]
        * vol_weights[m2]
        * rho_weights[drho]
    )

    weighted_loss = weight * loss

    if weighted_loss > max_weighted_loss:
        max_weighted_loss = weighted_loss
        worst_portfolio_loss = loss
        worst_weight = weight
        worst_weighted_scenario = (
            x1, x2, m1, m2, drho
        )

print("Maximum weighted scenario loss:", max_weighted_loss)
print("Worst weighted scenario:", worst_weighted_scenario)
print("Scenario weight:", worst_weight)
print("Portfolio loss:", worst_portfolio_loss)

Maximum weighted scenario loss: 1313.785913160907
Worst weighted scenario: (0.05, 0.05, 0.75, 0.75, 0.5)
Scenario weight: 1.953125
Portfolio loss: 672.6583875383844
